In [11]:
%pip install pandas optuna torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy

Note: you may need to restart the kernel to use updated packages.


In [12]:
import optuna
from collections import OrderedDict
from mllms import (
    JSONTuplesPromptTemplate,
    JsonDictPromptTemplate,
    PromptTemplate,
    LLMAddressParsingModel,
    LlamaAddressParsingModel,
    QwenAddressParsingModel
)
from optuna_utils import suggest_partial_permutation
import abc



In [13]:

ROLE_DESCRIPTIONS = OrderedDict([
    ("none", ""),
    (
        "german_archivist_ww2",
        "You are a German archivist handling the digitalization of German documents from "
        "the compensation efforts that followed the Second World War."
    ),
    (
        "archivist_ww2",
        "You are an archivist handling the digitalization of documents from "
        "the compensation efforts that followed the Second World War."
    ),
    (
        "german_archivist",
        "You are a German archivist handling the digitalization of German documents."
    ),
    (
        "archivist",
        "You are an archivist handling the digitalization of documents."
    )
])

TASK_DESCRIPTIONS = OrderedDict([
    ("simple", 
     "Annotate addresses, identifying the respective components of each address."),
    ("current_task", 
     "Your current task consists of annotating addresses "
     "identifying the respective components of each address."),
    ("archival_documents", 
     "Your current task consists of annotating addresses found in archival documents, "
     "identifying the respective components of each address.")
])

TASK_RESTRICTIONS = OrderedDict([
    (
        "loyal_to_text",
        "Remain loyal to the original text."
    ),
    (
        "explicitly_present_information", 
        "Only extract information **explicitly present in the address**."
    ),
    (
        "do_not_infer_missing_components",
        "Do **not infer missing components**."
    ),
    (
        "no_spelling_correction",
        "Do **not correct the spelling** of any word in the address."
    ),
    (
        "if_uncertain_exclude",
        "If uncertain about a component type, exclude it from the output."
    ),
    (
        "separate_neighborhood_and_city",
        "If the address contains a neighborhood joined together with a city "
        "by a dash (e.g., Berlin-Marienfelde), separate them accordingly."
    )
])

TASK_HINTS = OrderedDict([
    (
        "german_names", 
        "Addresses will often be written in German, meaning country and city names may be in German."
    ),
    (
        "hebrew",
        "Addresses in Israel will often have words in Hebrew."
    ),
    (
        "german_phonetic_interpretation",
        "It is likely that the address was written down by a German person "
        "who might have interpreted the phonetics of the words differently. "
        "Therefore, \"J\" might take the place of an \"I\" sound, "
        "\"W\" might take the place of a \"V\" sound, etc."
    )
])

EXAMPLE_TERMS = OrderedDict([
    (
        "burg", 
        "\"burg\" for city"
    ),
    (
        "stadt",
        "\"stadt\" for city"
    ),
    (
        "german_districts",
        "\"Kreis\" or its abbreviations \"Krs.\" or \"Kr.\" for district"
    ),
    (
        "straße",
        "straße or its abbreviation \"str.\" for street"
    ),
    (
        "avenue",
        "\"avenue\" or its abbreviation \"ave.\" for street"
    )
])

OUTPUT_FORMATS = OrderedDict([
    ("json_object", (
        "Format the output as a JSON object with the component types as keys.",
        JsonDictPromptTemplate
        )
    ),
    ("json_tuples", (
        "Format the output as a JSON list of [component, type] tuples.",
        JSONTuplesPromptTemplate
        )
    )
])

ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country",
    "Other"
]

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

OPTIONAL_ENTITIES = [entity for entity in ENTITIES if entity not in REQUIRED_ENTITIES]

PROMPT_SECTIONS = [
    "restrictions",
    "hints",
    "example_terms"
]

class PromptBuilder(abc.ABC):
    def __init__(
            self,
            role_description, 
            task_description,
            section_order,
            entities, 
            restrictions, 
            hints, 
            example_terms,
            terms_may_be_suffix,
            output_format
        ):
        self.role_description = role_description
        self.task_description = task_description
        self.section_order = section_order
        self.entities = entities
        self.restrictions = restrictions
        self.hints = hints
        self.example_terms = example_terms
        self.terms_may_be_suffix = terms_may_be_suffix
        self.output_format = output_format
        self.build_buffer : list[str] = []

    @abc.abstractmethod
    def _append_restrictions(self):
        raise NotImplementedError()
    
    @abc.abstractmethod
    def _append_hints(self):
        raise NotImplementedError()
    
    @abc.abstractmethod
    def _append_example_terms(self):
        raise NotImplementedError()

    def _append_item_list(self, keys : list[str], dictionary : dict[str, str]):
        for key in keys:
            self.build_buffer.append("- " + dictionary[key] + "\n")
    
    def _append_sections(self):
        for section in self.section_order:
            if section == "restrictions" and self.restrictions:
                self._append_restrictions()
            elif section == "hints" and self.hints:
                self._append_hints()
            elif section == "example_terms" and self.example_terms:
                self._append_example_terms()
            elif section not in PROMPT_SECTIONS:
                raise ValueError(f"Unknown section {section}.")

    @abc.abstractmethod
    def build(self) -> PromptTemplate:
        raise NotImplementedError()

class MarkdownPromptBuilder(PromptBuilder):
    def _append_restrictions(self):
        self.build_buffer.append("## Rules:\n\n")
        self._append_item_list(self.restrictions, TASK_RESTRICTIONS)
        self.build_buffer.append("\n")

    def _append_hints(self):
        self.build_buffer.append("## Hints:\n\n")
        self.build_buffer.append("When interpreting the addresses, take into consideration:\n")
        self._append_item_list(self.hints, TASK_HINTS)
        self.build_buffer.append("\n")

    def _append_example_terms(self):
        self.build_buffer.append("## Example Terms:\n\n")
        self.build_buffer.append("The addresses often include terms such as:\n")
        self._append_item_list(self.example_terms, EXAMPLE_TERMS)
        if self.terms_may_be_suffix:
            self.build_buffer.append("Some of these terms may occur as a suffix to another word.\n")
        self.build_buffer.append("\n")

    def build(self) -> PromptTemplate:
        self.build_buffer = []
        if self.role_description != "none":
            self.build_buffer.append("# Role\n\n")
            self.build_buffer.append(ROLE_DESCRIPTIONS[self.role_description])
            self.build_buffer.append("\n\n")
        self.build_buffer.append("# Task\n\n")
        self.build_buffer.append(TASK_DESCRIPTIONS[self.task_description])
        self.build_buffer.append(" Consider the component types: ")
        self.build_buffer.append(", ".join(self.entities))
        self.build_buffer.append(".\n\n")
        self._append_sections()
        self.build_buffer.append(OUTPUT_FORMATS[self.output_format][0])
        self.build_buffer.append("\n")
        self.build_buffer.append("%(examples)s\n")
        self.build_buffer.append("Now annotate the following address:\n%(address)s")
        prompt_template = "".join(self.build_buffer)
        example_prefix = "# Examples\n\n"
        prompt_class = OUTPUT_FORMATS[self.output_format][1]
        return prompt_class(
            template=prompt_template,
            examples_prefix=example_prefix
        )

class PlainPromptBuilder(PromptBuilder):
    def _append_restrictions(self):
        self.build_buffer.append("It is essential that while solving the task you stick to the following rules:\n")
        self._append_item_list(self.restrictions, TASK_RESTRICTIONS)
        self.build_buffer.append("\n")

    def _append_hints(self):
        self.build_buffer.append("When interpreting the addresses, take into consideration:\n")
        self._append_item_list(self.hints, TASK_HINTS)
        self.build_buffer.append("\n")

    def _append_example_terms(self):
        self.build_buffer.append("The addresses often include terms such as:\n")
        self._append_item_list(self.example_terms, EXAMPLE_TERMS)
        if self.terms_may_be_suffix:
            self.build_buffer.append("Some of these terms may occur as a suffix to another word.\n")
        self.build_buffer.append("\n")

    def build(self) -> PromptTemplate:
        self.build_buffer = []
        if self.role_description != "none":
            self.build_buffer.append(ROLE_DESCRIPTIONS[self.role_description])
            self.build_buffer.append(" ")
        self.build_buffer.append(TASK_DESCRIPTIONS[self.task_description])
        self.build_buffer.append(" Consider the component types: ")
        self.build_buffer.append(", ".join(self.entities))
        self.build_buffer.append(".\n\n")
        self._append_sections()
        self.build_buffer.append(OUTPUT_FORMATS[self.output_format][0])
        self.build_buffer.append("\n")
        self.build_buffer.append("%(examples)s\n")
        self.build_buffer.append("Now annotate the following address:\n%(address)s")
        prompt_template = "".join(self.build_buffer)
        example_prefix = "Consider the following examples:\n"
        prompt_class = OUTPUT_FORMATS[self.output_format][1]
        return prompt_class(
            template=prompt_template,
            examples_prefix=example_prefix
        )
    
PROMPT_FORMATS = OrderedDict([
    ("plain", PlainPromptBuilder),
    ("markdown", MarkdownPromptBuilder)
])

def suggest_prompt(trial : optuna.Trial, entities_to_predict) -> PromptTemplate:
    prompt_format = trial.suggest_categorical("prompt_format", ["plain", "markdown"])
    role_description = trial.suggest_categorical("prompt_role_description", list(ROLE_DESCRIPTIONS.keys()))
    task_description = trial.suggest_categorical("prompt_task_description", list(TASK_DESCRIPTIONS.keys()))
    restrictions = suggest_partial_permutation(trial, "prompt_restrictions", list(TASK_RESTRICTIONS.keys()))
    hints = suggest_partial_permutation(trial, "prompt_hints", list(TASK_HINTS.keys()))
    example_terms = suggest_partial_permutation(trial, "prompt_example_terms", list(EXAMPLE_TERMS.keys()))
    section_order = suggest_partial_permutation(trial, "prompt_section_order", PROMPT_SECTIONS)
    if len(example_terms) > 0:
        terms_may_be_suffix = trial.suggest_categorical("prompt_example_terms_may_be_suffix", [True, False])
    else: terms_may_be_suffix = False
    output_format = trial.suggest_categorical("output_format", list(OUTPUT_FORMATS.keys()))
    prompt_builder = PROMPT_FORMATS[prompt_format](
        role_description, task_description, section_order, entities_to_predict, restrictions,
        hints, example_terms, terms_may_be_suffix, output_format
    )
    return prompt_builder.build()

In [14]:
def get_random_prompts(n = 1, seed=None):
    sampler = optuna.samplers.RandomSampler(seed)
    study = optuna.create_study(sampler=sampler)
    return [
        suggest_prompt(study.ask(), ENTITIES) for _ in range(n)
    ]

for i, prompt in enumerate(get_random_prompts(n=5)):
    print(f"Prompt {i}")
    print(prompt.template)
    print("\n")

[I 2026-03-19 09:12:18,525] A new study created in memory with name: no-name-d3044a90-92b3-4c73-9dac-63ac16a38272


Prompt 0
# Role

You are an archivist handling the digitalization of documents from the compensation efforts that followed the Second World War.

# Task

Your current task consists of annotating addresses found in archival documents, identifying the respective components of each address. Consider the component types: HouseNumber, StreetName, Neighborhood, City, District, State, Region, Country, Other.

## Hints:

When interpreting the addresses, take into consideration:
- Addresses in Israel will often have words in Hebrew.
- It is likely that the address was written down by a German person who might have interpreted the phonetics of the words differently. Therefore, "J" might take the place of an "I" sound, "W" might take the place of a "V" sound, etc.

## Example Terms:

The addresses often include terms such as:
- straße or its abbreviation "str." for street
- "avenue" or its abbreviation "ave." for street
Some of these terms may occur as a suffix to another word.

Format the output

In [15]:
def suggest_example_strategy(trial : optuna.Trial, n_shots : int, entities_to_predict):
    example_strategy = trial.suggest_categorical("example_strategy", [
        "similar_semantic_embeddings",
        #"fixed_examples"
    ])
    if example_strategy == "similar_semantic_embeddings":
        threshold = trial.suggest_float("similar_semantic_embeddings_threshold", -1.0, 1.0, step=0.05)
        

def run_llama_trial(trial : optuna.Trial):
    entities = suggest_partial_permutation(trial, "prompt_extra_entities", OPTIONAL_ENTITIES)
    entities = REQUIRED_ENTITIES + entities
    entities = [entity for entity in ENTITIES if entity in entities] # Recover original order

    prompt = suggest_prompt(trial, entities)
    
    n_shots = trial.suggest_int("number_of_examples", 0, 10)
    if n_shots > 0:
        example_strategy = suggest_example_strategy(trial, n_shots, entities)
